**Lab 11: Data Exploration and Model Development**

*In this lab, we will pull in the class data that has been collated for sitting, standing, walking, and falling. After exploring the data, we will figure out how to best create an ML model that will take in gyroscope/accelerometer data from the Raspberry Pi and determine what behavior is being performed.*

**Part I. Set-Up**

First, we need to make sure we are working with a version of Tensorflow that complements our capabilities on our embedded system.

The Pi runs Python 3.7. We will upgrade to use a tflite_runtime interpreter == 2.11.0 BUT we will still have to downgrade tensorflow (for model development) to 2.12.0 so that the oeprations are compatible for inference on our embedded system.

In [ ]:
#Change if you need to
!pip uninstall scikit-learn
!pip install scikit-learn==1.0.1

In [ ]:
!pip uninstall tensorflow
!pip install tensorflow==2.12.0

In [ ]:
#Make sure it is 2.12.0 before doing any work
import tensorflow as tf
print(tf.__version__)

Next we need to mount our Drive and map to where our data lives. Please make sure that the CSV files given in the github repository are saved in a folder of your choosing on your Google Drive. Then map to that directory. Feel free to use

```
!ls
```

 to check that your files are there!

In [ ]:
# prompt: navigate to a folder in my google drive My Drive > data > fall_data

import os
from google.colab import drive

drive.mount('/content/drive')
%cd /content/drive/My Drive/data/fall_data
!pwd


In [ ]:
!ls

**Part II. Data Exploration**

Now let's pull in the data using


```
pd.read_csv()
```
Label the columns and explore this data set.


*   How many data points are there in each?
*   Maybe find a way to plot a couple of the parameters over time to see what the "behavior" looks like?




In [ ]:
import pandas as pd

# Load the four CSV files into dataframes without the index
df_sitting = pd.read_csv('sitting_data.csv', index_col=False)
df_standing = pd.read_csv('standing_data.csv', index_col=False)
df_walking = pd.read_csv('walking_data.csv', index_col=False)
df_falling = pd.read_csv('falling_data.csv', index_col=False)

# Specify the columns that contain gyro and accel data
gyro_accel_columns = ['Gyro_X', 'Gyro_Y', 'Gyro_Z', 'Accel_X', 'Accel_Y', 'Accel_Z']  # Replace with actual column names



In [ ]:
# data exploration here.

**Part III. Initial Model Development**
Now start to develop a model that may be able to successfully classify these behaviors. We will start with just using the raw data as input.

We also have to think about what libraries will be using and what versions are available for our Raspberry Pi. If you are using Scikit, make sure you are compatible!

In [ ]:
pip show scikit-learn

In [ ]:
# Import libraries (add libraries you need here)
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout,Conv1D, MaxPooling1D, Flatten, Dense
from tensorflow.keras.utils import to_categorical
from sklearn import svm
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

You may want to concatenate all of your dataframes into one dataframe.

In [ ]:
# Concatenate all windowed dataframes into one large dataframe X
X = pd.concat([df_sitting, df_standing, df_walking, df_falling], ignore_index=True)

# Randomize the order of the X data
X = shuffle(X, random_state=42)
# Separate the labels into a dataframe y
y = X.pop('Label')

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_categorical = to_categorical(y_encoded, num_classes=4)  # Ensure 4 classes for one-hot encoding

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X[gyro_accel_columns], y_encoded, test_size=0.2, random_state=42)

# Create a Random Forest classifier
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42) # You can adjust hyperparameters

# Fit the classifier to the training data
rf_classifier.fit(X_train, y_train)

In [ ]:
# Predict the labels for the test set
y_pred = rf_classifier.predict(X_test)

# Calculate the accuracy of the classifier
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy of the SVM classifier: {accuracy}")

At this point there may be some optimization that you wish to do. For time's sake, we are going to skip this to demonstrate the on-system testing.

In [ ]:
#Optimization of model here.

**Part IV. Model Compression/Export**

Now, we need to save our model so that it will work on the Raspberry Pi.

Scikit-Learn does not offer model compression. We can compare the sizes of these models with other NN models we will build later.

We will use TensorFlow Lite to compress our NN model.

In [ ]:
#Set-up
model = rf_classifier
model_file_name = 'rf_model.pkl'
import pickle

# Save the model to a file
filename = model_file_name
pickle.dump(model, open(filename, 'wb'))



**PART V. Inference**

Once that is done, you need to set up a script that takes in information from the Raspberry Pi/SenseHat sensors, applies the model and reports the inference/prediction.

The script below takes in the raw data from the SenseHat, loads the model, makes a prediction, and indicates the prediction as sa color on the SenseHat:





*   BLUE = sitting
*   YELLOW = standing

*   GREEN = walking
*   RED = falling





In [ ]:
# ON Raspberry Pi Script!
# This script should be saved as rf_predict.py on your device.
# It should be in the same directory as your rf_model.pkl file.
# YOU NEED A PYTHON ENVIRONMENT SET UP BEFORE RUNNING THIS!

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import pickle
from sense_hat import SenseHat

# Initialize SenseHAT
sense = SenseHat()

# Load the trained model
model_file_name = 'rf_model.pkl'
loaded_model = pickle.load(open(model_file_name, 'rb'))

# Function to map activity to color
def get_color(activity):
    if activity == 'sitting':
        return (0, 0, 255)  # Blue
    elif activity == 'standing':
        return (255, 255, 0) # Yellow
    elif activity == 'walking':
        return (0, 255, 0) # Green
    elif activity == 'falling':
        return (255, 0, 0) # Red
    else:
        return (255, 255, 255) # Default to white

# Function to get data from sensehat in the correct form for the model
def get_sensehat_data():
    gyro = sense.get_gyroscope_raw()
    accel = sense.get_accelerometer_raw()
    data = np.array([gyro['x'], gyro['y'], gyro['z'], accel['x'], accel['y'], accel['z']])
    return data.reshape(1, -1)  # Reshape for model input

try:
    while True:
        # Get sensor data
        input_data = get_sensehat_data()

        # Make a prediction
        prediction = loaded_model.predict(input_data)

**Part VI. Tensorflow**

Now let's use NN. We will start with a simple NN that takes in raw data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Create a neural network model
model = Sequential()
model.add(Dense(128, input_dim=X_train.shape[1], activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(4, activation='softmax'))  # Assuming 4 classes

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=25, batch_size=32, validation_split=0.2)

# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Neural Network accuracy: {accuracy:.2f}")




In [ ]:
#Set-up
model = model
model_file_name = 'nn_model_raw.tflite'
#Convert the model to TensorFlow Lite format
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

#Save the TensorFlow Lite model to a file
with open(model_file_name, 'wb') as f:
    f.write(tflite_model)

print("Model converted to TensorFlow Lite and saved as ", model_file_name)

In [ ]:
# ON Raspberry Pi Script!
# This script should be saved as nn_predict.py on your device.
# It should be in the same directory as your nn_model_raw.tflite file.
# YOU NEED A PYTHON ENVIRONMENT SET UP BEFORE RUNNING THIS!
import numpy as np
from sense_hat import SenseHat
import tflite_runtime.interpreter as tflite

# Initialize SenseHat
sense = SenseHat()

# Load the TensorFlow Lite model
interpreter = tflite.Interpreter(model_path="model.tflite")
interpreter.allocate_tensors()

# Get input and output tensors
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Function to get gyro/accel data from SenseHat
def get_gyro_accel_data():
    accel = sense.get_accelerometer_raw()
    gyro = sense.get_gyroscope_raw()
    data = np.array([accel['x'], accel['y'], accel['z'], gyro['x'], gyro['y'], gyro['z']])
    return data

# Function to make a prediction based on gyro/accel data
def predict(data):
    # Preprocess the data to match the input shape of the model
    data = data.reshape((1, 5, 6)).astype(np.float32)

    # Set the tensor to point to the input data to be inferred
    interpreter.set_tensor(input_details[0]['index'], data)

    # Run inference
    interpreter.invoke()

    # Get the prediction results
    output_data = interpreter.get_tensor(output_details[0]['index'])
    prediction = np.argmax(output_data)

    return prediction

# Function to update LED matrix based on prediction
def update_led_matrix(prediction):
    if prediction == 0:
        color = (0, 0, 255)  # Blue for sitting
        print("Sitting")
    elif prediction == 1:
        color = (255, 255, 0)  # Yellow for standing
        print("Standing")
    elif prediction == 2:
        color = (0, 255, 0)  # Green for walking
        print("Walking")
    elif prediction == 3:
        color = (255, 0, 0)  # Red for falling

**Part VII. A More Sophisticated NN**

As we noticed, a simple NN with raw values did not perform very well. Let's try using window of time to break up our sensor data and use THOSE as inputs to our model.

In [ ]:
# Function to create a sliding window dataframe for specific columns
def create_sliding_window_df(df, columns, window_size):
    sliding_window_data = []
    for i in range(len(df) - window_size + 1):
        window = df[columns].iloc[i:i + window_size].values.flatten()
        sliding_window_data.append(window)
    return pd.DataFrame(sliding_window_data)

In [ ]:
# Define your window_size
window_size = 100

# Create new dataframes with sliding windows for gyro and accel data
df_sitting_windowed = create_sliding_window_df(df_sitting, gyro_accel_columns,window_size)
df_standing_windowed = create_sliding_window_df(df_standing, gyro_accel_columns,window_size)
df_walking_windowed = create_sliding_window_df(df_walking, gyro_accel_columns,window_size)
df_falling_windowed = create_sliding_window_df(df_falling, gyro_accel_columns,window_size)

# Add labels to the windowed dataframes
df_sitting_windowed['Label'] = 0
df_standing_windowed['Label'] = 1
df_walking_windowed['Label'] = 2
df_falling_windowed['Label'] = 3

In [ ]:



# Load the four CSV files into dataframes without the index
df_sitting_windowed = pd.read_csv('sitting_data_windowed.csv', index_col=False)
df_standing_windowed = pd.read_csv('standing_data_windowed.csv', index_col=False)
df_walking_windowed = pd.read_csv('walking_data_windowed.csv', index_col=False)
df_falling_windowed = pd.read_csv('falling_data_windowed.csv', index_col=False)

# Concatenate all windowed dataframes into one large dataframe X
X = pd.concat([df_sitting_windowed, df_standing_windowed, df_walking_windowed, df_falling_windowed], ignore_index=True)

# Randomize the order of the X data
X = shuffle(X, random_state=42)
# Separate the labels into a dataframe y
y = X.pop('Label')

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_categorical = to_categorical(y_encoded, num_classes=4)  # Ensure 4 classes for one-hot encoding


In [ ]:
# Reshape X to be 3D [samples, time steps, features]
time_steps = 100  # This should match the sliding window size
n_features = X.shape[1] // time_steps
X = X.values.reshape((X.shape[0], time_steps, n_features))

# Split the data into training, testing, and validation sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y_categorical, test_size=0.4, random_state=42)
X_test, X_val, y_test, y_val = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print("Data has been successfully split into training, testing, and validation sets.")
print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")
print(f"Validation set size: {len(X_val)}")

# Build the CNN model
model = Sequential()
model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(time_steps, n_features)))
model.add(MaxPooling1D(pool_size=2))
model.add(Flatten())
model.add(Dense(100, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(4, activation='softmax'))  # Assuming 4 classes: sitting, standing, walking, falling

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_val, y_val))

# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Test Accuracy: {accuracy:.2f}')

In [ ]:
#Set-up
model = model
model_file_name = 'nn_window_model_raw.tflite'

# Convert the model to TensorFlow Lite format
converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,  # Built-in operations
    tf.lite.OpsSet.SELECT_TF_OPS     # Select TensorFlow operations
]
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.inference_input_type = tf.float32
converter.inference_output_type = tf.float32
tflite_model = converter.convert()

# Save the TensorFlow Lite model to a file
with open(model_file_name, 'wb') as f:
    f.write(tflite_model)

print("Model converted to TensorFlow Lite and saved as ", model_file_name)
